# 09｜保存证据、数据库与私有样本

目标：将原始图摘要、参数、质量、结果、原因一起保存，并避免重复写入同一事件。用 SQLite 演示本地存储，不需要数据库服务器或口令。现场系统可以再对接 PostgreSQL/MySQL 和 MES，但不能把“放进待同步表”显示成“MES 已接收”。

下方记录明确为教学规则输入，来源是公开照片；不伪装成实时检测器结果。数据库文件被 Git 忽略，执行笔记本不会连接任何远程系统。

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab_utils.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import lab_utils as lab
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

OpenCV 4.12.0 | NumPy 2.2.6


In [2]:
source = lab.case_row("weld_good")
image = lab.load_case("weld_good")
quality = lab.image_quality(image)
result = lab.seven_slot_verdict(["OK"] * 5 + ["NG"] * 2)
result["basis"] = "synthetic_rule_input_not_visual_inference"
recipe = {"name": "teaching-rule-v1", "required_good": 5, "total_slots": 7,
          "unknown_policy": "REVIEW", "lighting_actual": "unknown"}
database = lab.OUTPUT / "teaching_inspections.sqlite3"
connection = lab.create_database(database)
event_key = "teaching-v1:" + source["sha256"] + ":5-of-7-synthetic-rule"
record_id, inserted = lab.save_inspection(connection, event_key=event_key,
    source_id=source["id"], source_sha256=source["sha256"], evidence_path=source["file"],
    result=result, recipe=recipe, quality=quality)
same_id, inserted_twice = lab.save_inspection(connection, event_key=event_key,
    source_id=source["id"], source_sha256=source["sha256"], evidence_path=source["file"],
    result=result, recipe=recipe, quality=quality)
assert record_id == same_id and inserted_twice is False
print("记录 ID", record_id, "首次新增", inserted, "重复提交新增", inserted_twice)

记录 ID 1 首次新增 False 重复提交新增 False


## 1. 结果表与待同步表在同一事务中写入

本地结果和 outbox 一起提交，减少“结果已存但待发送事件丢失”的情况。发送服务需要用 `event_key` 做幂等、确认服务端回执后才改 `synced`；网络失败可以重试。这里没有 MES 地址，也不模拟成功回执。

实际实时视频的 `event_key` 应包含采集会话和帧/工件标识，不能只用图像摘要，否则两次合法检测可能被误合并。

In [3]:
print("检测结果", connection.execute("SELECT id,source_id,status,observed FROM inspections").fetchall())
print("真实同步状态", connection.execute("SELECT inspection_id,state,attempts FROM sync_outbox").fetchall())
assert connection.execute("SELECT state FROM sync_outbox WHERE inspection_id=?", (record_id,)).fetchone()[0] == "pending"
connection.close()
public_record = {"source": source["id"], "source_sha256": source["sha256"],
                 "quality": quality, "recipe": recipe, "result": result,
                 "sync": {"state": "pending", "remote_ack": None}}
(lab.OUTPUT / "teaching_inspection.json").write_text(json.dumps(public_record, ensure_ascii=False, indent=2), encoding="utf-8")
display(public_record)

检测结果 [(1, 'weld_good', 'OK', 1)]
真实同步状态 [(1, 'pending', 0)]


{'source': 'weld_good',
 'source_sha256': 'a57d4e8615c3c87e0cd6b522fd84a21e4e39e55265907b8cd49589a7a4160f1b',
 'quality': {'brightness': 48.829,
  'sharpness': 207.265,
  'dark_ratio': 0.33159,
  'overexposed_ratio': 0.00843,
  'width': 640,
  'height': 640},
 'recipe': {'name': 'teaching-rule-v1',
  'required_good': 5,
  'total_slots': 7,
  'unknown_policy': 'REVIEW',
  'lighting_actual': 'unknown'},
 'result': {'good_count': 5,
  'required_good': 5,
  'total_slots': 7,
  'slots': ['OK', 'OK', 'OK', 'OK', 'OK', 'NG', 'NG'],
  'status': 'OK',
  'reason': '七格外观中5格符合已确认标准',
  'basis': 'synthetic_rule_input_not_visual_inference'},
 'sync': {'state': 'pending', 'remote_ack': None}}

## 2. 本地导入私有刀片电池样本

默认课程从不读取 `data/private`，公开网页只包含上述许可明确的公开照片。需要练现场九块汇流排时，在终端显式导入自己的图片，例如：

```powershell
conda run -n pyg python scripts/import_private.py "D:\your-private-folder\battery.jpg" --expected-count 9
```

这里的 9 必须来自人工确认，不是算法成绩。导入器把像素重新编码成 PNG、去掉原 EXIF，把 SHA256 和人工数量保存在 `data/private`；七格真值仍为未知。该目录和 `outputs/private` 均被 Git 忽略。请在私有笔记本另行处理，不把私人输出写回公开教材。

实时采集还需要真实摄像头服务，不能用轮播照片冒充。采集线程只保留最新帧；检测线程处理完成后再取新帧；图像、检测框和 frame_id 原子切换。设备不支持对焦/曝光回读时，数据库字段保留空值。

In [4]:
# 不读取私有目录；只确认默认数据入口确实是公开清单。
assert all(not row["file"].startswith("data/private/") for row in lab.manifest()["files"])
print("默认课程不会导出私有像素；真实相机、MES 和七格视觉识别尚需各自现场验收。")

默认课程不会导出私有像素；真实相机、MES 和七格视觉识别尚需各自现场验收。


## 最后的复现检查

重新执行本课，同一事件不应重复插入；换成新的会话事件应正常新增。打开 `outputs/public/teaching_inspection.json`，检查结果理由、来源摘要和 pending 状态。完成课程表示掌握一套可运行的图像与数据流程，不等于工业产品已达到任何准确率要求。

## 来源与练习边界

公开焊缝：sukmaadhiwijaya / rikkarth，固定提交 `8b80d1afe88c0b946500a498cbdb9d7a1eda5533`，数据卡声明 CC0-1.0；工业表面：Kolektor Group / ViCoS，CC BY-NC-SA 4.0；分散零件：Tung1213，CC BY-SA 3.0；重叠零件：Technologyfelix，CC BY-SA 4.0。相关照片和派生图保留原许可及署名。

完整来源、原许可页、下载路径及 SHA256 见 [DATA_SOURCES.md](../DATA_SOURCES.md) 和 [source_manifest.json](../sources/source_manifest.json)。本课是非商业教学。灰度、二值、叠框及扰动均为原照片的派生图，七格合成图另有醒目标记。它们不等于新增独立验收数据，也不证明电池焊接准确率或机械强度。